# Turtle Breakout 18 — diagnostic: is BTC mean-reverting while the turtle is out of the market?

Pre-registration: `discussions/Turtle_Breakout/020_claude_tb18_preregistration.md`. Code: `turtle_flat_diag.py`; data: the TB 17 BTC-USD daily file.

Day $t$ is **IN** if the TB 17 turtle (long only, TB 01 rules) holds a long at its close, **FLAT** otherwise. On FLAT days: variance ratios of the next $q$ days ($q$ = 2, 5, 10, 20), autocorrelations, the reversal spreads (top − bottom quintile of the past 1- and 5-day return → next 1 and 5 days) and a z-score table, against 2,000 permutations of the returns.
**MR_PRESENT** only if some VR(q) < 1 with p < 0.0125 **and** a reversal spread (1-day or 5-day) < 0 with p < 0.025.

Environment: `TB17_DATA` (default `<parent of DERIBIT_DATA_ROOT>/Stocks/tb17/btc_daily.parquet`), `TB18_SYNTHETIC=1`.

In [ ]:
import os, sys, json, subprocess, tempfile, warnings, platform
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'data_engineering').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); load_dotenv(ROOT / '.env')
from data_engineering.modeling import turtle_breakout as tb, turtle_reverse as tr, turtle_stocks as tk, turtle_stock_accounts as ta, turtle_btc_history as bh, turtle_flat_diag as fd

SYNTHETIC = os.getenv('TB18_SYNTHETIC', '0') == '1'
DP = fd.DParams(); BP = bh.BParams()
P = replace(tb.Params(), min_bar_share=0.0, funding=False, fee=BP.side_cost - tb.Params().slip); RP = replace(tr.RParams(), side_cost=BP.side_cost)
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='tb18-')) / '18'; DP = replace(DP, n_perm=200)
    d, _ = tk.synthetic_daily(4400, seed=18, vol=0.04, gap=0.0, steps=48, drift=0.0005); d.index = pd.date_range('2014-09-17', periods=len(d), freq='D'); DAILY = d
else:
    OUT = ROOT / 'notebooks' / 'Turtle_Breakout' / '18'
    root = Path(os.environ['DERIBIT_DATA_ROOT']).parent
    DATA_FILE = Path(os.environ.get('TB17_DATA', str(root / 'Stocks' / 'tb17' / 'btc_daily.parquet')))
    if not DATA_FILE.exists():
        sys.path.insert(0, str(ROOT / 'tools')); import download_tb17_btc as dl; DATA_FILE = dl.download(DATA_FILE.parent)
    RAW = pd.read_parquet(DATA_FILE); RAW['date'] = pd.to_datetime(RAW['date']).dt.tz_localize(None).dt.normalize()
    DAILY = RAW.set_index('date')[['open', 'high', 'low', 'close']].sort_index(); DAILY = DAILY[~DAILY.index.duplicated()].dropna(); DAILY = DAILY[(DAILY > 0).all(axis=1)]
OUT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy(); env['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + env.get('PYTHONPATH', '')
chk = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_turtle_flat_diag.py'], cwd=str(ROOT), env=env, capture_output=True, text=True)
print(chk.stderr.strip().splitlines()[-1]); assert chk.returncode == 0
print('BTC-USD', DAILY.index[0].date(), '->', DAILY.index[-1].date(), len(DAILY)); print(json.dumps(fd.registration(DP), indent=1))

## 1. Turtle state and trend labels

In [ ]:
W = bh.run_window(DAILY, '2014-09-17', None, P, RP, ta.AParams(), 0, np.random.default_rng(0))
TR = W['trades'].query("kind == 'long'").copy()
TR.loc[TR.open_at_end, 'exit'] = DAILY.index[-1] + pd.Timedelta(days=1)
start = W['long'].index[0]
close = DAILY.close; r = np.log(close).diff().to_numpy()
STATE = fd.turtle_state(DAILY.index, TR, start); TREND = fd.trend_label(close)
valid = STATE.notna().to_numpy() & np.isfinite(r)
MASKS = {'all': valid, 'in': valid & (STATE == 'in').to_numpy(), 'flat': valid & (STATE == 'flat').to_numpy()}
for lab in ('down', 'range', 'up'):
    MASKS[f'flat_{lab}'] = MASKS['flat'] & (TREND == lab).to_numpy()
COMP = pd.DataFrame({'days': {k: int(m.sum()) for k, m in MASKS.items()}}); COMP['share_of_all'] = COMP.days / COMP.loc['all', 'days']
display(COMP.round(3))
sp = (STATE == 'flat').astype(int); runs = sp.groupby((sp != sp.shift()).cumsum()).agg(['first', 'size']); fl = runs[runs['first'] == 1]['size']
print(f"flat spells: {len(fl)}, median {fl.median():.0f} days, mean {fl.mean():.0f}, longest {fl.max()}")
print('flat days by year:'); display((STATE == 'flat').groupby(DAILY.index.year).mean().round(2).to_frame('share_flat').T)

## 2. Statistics by state, with permutation nulls

In [ ]:
STATS = {}
for k, m in MASKS.items():
    null = fd.permutation(r, m, replace(DP, seed=DP.seed + len(STATS)))
    STATS[k] = fd.state_stats(r, m, null)
ST = pd.DataFrame(STATS).T
display(ST[['days', 'ann_drift_pct', 'ann_vol_pct'] + [f'VR{q}' for q in fd.QS] + [f'p_VR{q}_below' for q in fd.QS] + ['REV11', 'p_REV11_below', 'REV55', 'p_REV55_below']].round(4))
display(ST[[f'AC{k}' for k in range(1, 11)]].round(3))
print('approx. 95% band for one AC on FLAT days: +/-', round(1.96 / np.sqrt(ST.loc['flat', 'days']), 3))

## 3. Decision (FLAT days)

In [ ]:
DEC = fd.decide(STATS['flat'], DP)
DEC['by_trend_reported'] = {k: fd.decide(STATS[k], DP) for k in ('flat_down', 'flat_range', 'flat_up')}
print(json.dumps(DEC, indent=1, default=float))

## 4. Reported: reversal and z-score tables

In [ ]:
for k in ('flat', 'in', 'flat_range', 'flat_down'):
    for kk in (1, 5):
        s, T = fd.rev(r, MASKS[k], kk, kk); print(f'{k}: next-{kk}-day log return by past-{kk}-day quintile (spread {s:+.4f})'); display(T.round(4))
for k in ('flat', 'in'):
    print(f'{k}: next-5-day log return by z20 bucket'); display(fd.z_table(close, r, MASKS[k], DP.z_window, 5).round(4))

## 5. Figures

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [2, 1]})
axes[0].plot(close.index, close, color='k', lw=0.8); axes[0].set_yscale('log')
inn = (STATE == 'in').to_numpy()
axes[0].fill_between(close.index, close.min(), close.max(), where=inn, color='#1f5aa6', alpha=0.15, label='turtle IN')
axes[0].legend(frameon=False); axes[0].set_title('BTC-USD and the days the turtle holds a long', loc='left', fontsize=10)
for k, col in (('all', '0.5'), ('in', '#1f5aa6'), ('flat', '#c2571a'), ('flat_range', '#3a7d44'), ('flat_down', '#b33f40')):
    axes[1].plot(fd.QS, [STATS[k][f'VR{q}'] for q in fd.QS], marker='o', color=col, label=k)
axes[1].axhline(1, color='k', lw=0.7); axes[1].set_xlabel('q (days)'); axes[1].set_ylabel('variance ratio'); axes[1].legend(frameon=False, fontsize=8, ncol=5)
for a in axes: a.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 6. Save

In [ ]:
ST.to_csv(OUT / '18_stats_by_state.csv'); COMP.to_csv(OUT / '18_composition.csv')
pd.DataFrame({'state': STATE, 'trend': TREND}).to_csv(OUT / '18_state_daily.csv')
(OUT / '18_decision.json').write_text(json.dumps({'decision': DEC, 'registration': fd.registration(DP), 'synthetic': SYNTHETIC}, indent=1, default=float))
git = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=str(ROOT), capture_output=True, text=True).stdout.strip()
(OUT / '18_manifest.json').write_text(json.dumps({'version': fd.VERSION, 'git': git, 'python': platform.python_version(), 'created_utc': pd.Timestamp.now('UTC').isoformat(), 'synthetic': SYNTHETIC}, indent=1))
print('saved to', OUT); print('VERDICT:', DEC['verdict'])